# Rossmann Sales: EDA and Logistic Regression (Ridge and Lasso)

## 1. Load data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")

data_train = pd.read_csv('train.csv', low_memory=False)
data_store = pd.read_csv('store.csv')
data_test = pd.read_csv('test.csv')

# one row in store.csv has a blank Store id, the only id missing from 1..1115 is 490
data_store['Store'] = data_store['Store'].fillna(490).astype(int)

full_train = pd.merge(data_train, data_store, how='left', on='Store')
full_test = pd.merge(data_test, data_store, how='left', on='Store')

print(full_train.head())

## 2. Missing values

In [ ]:
print(full_train.isnull().sum())
print(full_test.isnull().sum())

In [ ]:
for df in [full_train, full_test]:
    df['Open'] = df['Open'].fillna(1)
    df[['Promo2SinceWeek', 'Promo2SinceYear']] = df[['Promo2SinceWeek', 'Promo2SinceYear']].fillna(0)
    df['PromoInterval'] = df['PromoInterval'].fillna('None')
    df['StateHoliday'] = df['StateHoliday'].astype(str)

    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day

# CompetitionDistance and CompetitionOpenSince* are still missing, the pipeline fills them below
print(full_train.isnull().sum()[full_train.isnull().sum() > 0])

## 3. EDA

In [ ]:
open_days = full_train[full_train['Open'] == 1]

print("Closed days:", (full_train['Open'] == 0).sum(), "(Sales is always 0)")
print("Sales skewness (open days):", open_days['Sales'].skew())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(full_train['Sales'], bins=50, ax=axes[0])
axes[0].set_title('Sales, all days')
sns.histplot(open_days['Sales'], bins=50, kde=True, ax=axes[1])
axes[1].set_title('Sales, open days')
plt.show()

In [ ]:
cols = ['DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment']

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, cols):
    open_days.groupby(col)['Sales'].mean().plot(kind='bar', ax=ax, title='Mean Sales by ' + col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(open_days.select_dtypes(include=['number']).drop(columns=['Store', 'Open']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 4. Target and split

Logistic regression needs classes, so HighSales = 1 if Sales is above the median, else 0. Closed days are removed. Customers is dropped because it is not in test.csv.

In [ ]:
model_data = full_train[full_train['Open'] == 1].copy()
model_data['HighSales'] = (model_data['Sales'] > model_data['Sales'].median()).astype(int)

num_cols = ['Promo', 'SchoolHoliday', 'Promo2', 'CompetitionDistance', 'CompetitionOpenSinceMonth',
            'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear', 'Year', 'Day']
cat_cols = ['DayOfWeek', 'Month', 'StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']

x = model_data[num_cols + cat_cols].copy()
x[cat_cols] = x[cat_cols].astype(str)
y = model_data['HighSales']

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(x_train.shape, x_val.shape)

## 5. Pipeline

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

## 6. Ridge (L2)

In [ ]:
param_grid = {'model__C': [0.01, 0.1, 1, 10]}

ridge_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(penalty='l2', solver='liblinear'))
])

ridge_grid = GridSearchCV(ridge_pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
ridge_grid.fit(x_train, y_train)

print("Best C:", ridge_grid.best_params_['model__C'])
print("Best CV accuracy:", ridge_grid.best_score_)

## 7. Lasso (L1)

In [ ]:
lasso_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(penalty='l1', solver='liblinear'))
])

lasso_grid = GridSearchCV(lasso_pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
lasso_grid.fit(x_train, y_train)

print("Best C:", lasso_grid.best_params_['model__C'])
print("Best CV accuracy:", lasso_grid.best_score_)

## 8. Compare

In [ ]:
models = {'Ridge': ridge_grid, 'Lasso': lasso_grid}

for name, grid in models.items():
    pred = grid.predict(x_val)
    print(name)
    print("  Accuracy:", round(accuracy_score(y_val, pred), 4))
    print("  F1:      ", round(f1_score(y_val, pred), 4))
    print("  ROC AUC: ", round(roc_auc_score(y_val, grid.predict_proba(x_val)[:, 1]), 4))
    print("  Confusion matrix:")
    print(confusion_matrix(y_val, pred))

## 9. Predict on test.csv

In [ ]:
best_model = ridge_grid if ridge_grid.score(x_val, y_val) >= lasso_grid.score(x_val, y_val) else lasso_grid

x_test = full_test[num_cols + cat_cols].copy()
x_test[cat_cols] = x_test[cat_cols].astype(str)

test_pred = best_model.predict(x_test)
test_pred[full_test['Open'].values == 0] = 0     # closed stores have no sales

submission = pd.DataFrame({'Id': full_test['Id'], 'HighSales': test_pred})
submission.to_csv('submission.csv', index=False)
print(submission.head())